In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import RobertaTokenizer, RobertaModel

# === Paths ===
BASE_DIR   = Path(r"D:\Shikha\bias_project")
OUTPUT_DIR = BASE_DIR / "outputs" / "embeddings" / "roberta"
WORD_LISTS = BASE_DIR / "word_lists" / "caliskan_2017"
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load RoBERTa ===
print("Loading RoBERTa model and tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model     = RobertaModel.from_pretrained('roberta-base')
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)
print(f"Model loaded on  : {device}")
print(f"Hidden size      : {model.config.hidden_size}")

# === Load word lists (same as BERT notebook) ===
def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print(f"\nWEAT 6 word lists loaded:")
print(f"Career : {career_terms}")
print(f"Family : {family_terms}")
print(f"Male   : {male_names}")
print(f"Female : {female_names}")

d:\Shikha\bias_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading RoBERTa model and tokenizer...


d:\Shikha\bias_project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhavi\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6148.86it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
K

Model loaded on  : cpu
Hidden size      : 768

WEAT 6 word lists loaded:
Career : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male   : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']


In [2]:
# Load Garg pairs and Measure 2/3 lists for full SEAT coverage
male_pairs   = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs = load_word_list(GARG_LISTS / "female_pairs.txt")

def get_roberta_embedding(text, tokenizer, model, device):
    """Get RoBERTa <s> (CLS-equivalent) token embedding for a sentence"""
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=64, padding=True
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # RoBERTa: first token (<s>) as sentence representation
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def make_sentence(word):
    return f"This is {word}."

def seat_score(target_A_embs, target_B_embs, attr_C_embs, attr_D_embs):
    """SEAT: positive = group A (male) more associated with attr C."""
    def mean_cos(emb, attr_embs):
        return np.mean([cosine_sim(emb, a) for a in attr_embs.values()])
    def assoc(emb, C, D):
        return mean_cos(emb, C) - mean_cos(emb, D)
    a = [assoc(e, attr_C_embs, attr_D_embs) for e in target_A_embs.values()]
    b = [assoc(e, attr_C_embs, attr_D_embs) for e in target_B_embs.values()]
    return np.mean(a) - np.mean(b)

# Global RoBERTa SEAT baseline (WEAT6 career-family, template sentences)
career_embs = {w: get_roberta_embedding(make_sentence(w), tokenizer, model, device) for w in career_terms}
family_embs = {w: get_roberta_embedding(make_sentence(w), tokenizer, model, device) for w in family_terms}
male_embs   = {w: get_roberta_embedding(make_sentence(w), tokenizer, model, device) for w in male_names}
female_embs = {w: get_roberta_embedding(make_sentence(w), tokenizer, model, device) for w in female_names}

global_seat = seat_score(male_embs, female_embs, career_embs, family_embs)
print(f"Global RoBERTa SEAT (WEAT6 career-family): {global_seat:.4f}")
print(f"Direction: {'male->career' if global_seat > 0 else 'female->career'}")
print(f"(BERT global baseline was 0.0063 for comparison)")

Global RoBERTa SEAT (WEAT6 career-family): -0.0000
Direction: female->career
(BERT global baseline was 0.0063 for comparison)


In [3]:
import random
CORPUS_DIR = BASE_DIR / "corpus"

# Load corpora
print("Loading corpora...")
cr_df       = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")
ca_df       = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv")
gd_df       = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv")
gd_df       = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")
print("Done.")

EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

# Build per-(state, period) sentence lists
state_period_sentences = {}

def add_sentences(state_abbr, period, text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return
    sents = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 20]
    key = (state_abbr, period)
    state_period_sentences.setdefault(key, []).extend(sents)

for _, row in ca_df.iterrows():
    add_sentences(str(row['state_abbr']), 'hist_1920s', str(row.get('text','')))
for _, row in rd_posts.iterrows():
    add_sentences(str(row['state_abbr']), 'modern_2020', str(row.get('title',''))+'. '+str(row.get('selftext','')))
for _, row in rd_comments.iterrows():
    add_sentences(str(row['state_abbr']), 'modern_2020', str(row.get('body','')))
for _, row in cr_df.iterrows():
    add_sentences(str(row['state']), 'modern_2024', str(row.get('text','')))
for _, row in gd_df.iterrows():
    add_sentences(str(row['state']), 'modern_2024', str(row.get('text','')))

print(f"State-period cells: {len(state_period_sentences)}")

Loading corpora...
Done.
State-period cells: 111


In [5]:
import random
random.seed(42)
SENTENCES_PER_WORD = 5

# === Mean-pooled RoBERTa embedding (more discriminative than <s> token) ===
def get_roberta_embedding(text, tokenizer, model, device):
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=64, padding=True
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    attention_mask = inputs['attention_mask'].squeeze(0).unsqueeze(-1)
    summed = (token_embeddings * attention_mask).sum(0)
    counts = attention_mask.sum(0)
    return (summed / counts).cpu().numpy()

def get_contextual_embedding(word, sentences, n=5):
    matching = [s for s in sentences if word.lower() in s.lower()]
    if not matching:
        matching = [f"This is {word}."]
    sampled = random.sample(matching, min(n, len(matching)))
    embs = [get_roberta_embedding(s[:256], tokenizer, model, device) for s in sampled]
    return np.mean(embs, axis=0)

results_roberta = []

print(f"{'State':<7} {'Period':<13} {'Sents':>6} {'SEAT':>9} {'Direction'}")
print("-" * 50)

for (state_abbr, period) in sorted(state_period_sentences.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_sentences[(state_abbr, period)]
    if len(sentences) < 20:
        continue

    c  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in career_terms}
    f  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in family_terms}
    m  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in male_pairs}
    fe = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in female_pairs}

    score = seat_score(m, fe, c, f)
    direction = "male->career" if score > 0 else "female->career"
    print(f"{state_abbr:<7} {period:<13} {len(sentences):>6} {score:>9.4f} {direction}")
    results_roberta.append({
        'state': state_abbr, 'period': period,
        'n_sentences': len(sentences),
        'seat_career_family': round(score, 4),
        'method': 'roberta_seat_meanpool'
    })

print(f"\nCells scored: {len(results_roberta)}")

State   Period         Sents      SEAT Direction
--------------------------------------------------
AK      modern_2020      283   -0.0011 female->career
AK      modern_2024     1375   -0.0002 female->career
AL      modern_2020      271   -0.0038 female->career
AL      modern_2024      470   -0.0036 female->career
AR      modern_2020      383    0.0041 male->career
AR      modern_2024      307   -0.0033 female->career
AZ      modern_2020      275   -0.0014 female->career
AZ      modern_2024      790    0.0096 male->career
CA      hist_1920s       150    0.0011 male->career
CA      modern_2020      340    0.0053 male->career
CA      modern_2024     1783    0.0000 male->career
CO      modern_2020      254   -0.0007 female->career
CO      modern_2024      461   -0.0014 female->career
CT      modern_2020      378   -0.0011 female->career
CT      modern_2024      463    0.0041 male->career
DE      modern_2020      452   -0.0002 female->career
DE      modern_2024      347    0.0096 male->car

In [6]:
roberta_df = pd.DataFrame(results_roberta).sort_values(['period', 'state'])

OUT_ROBERTA = OUTPUT_DIR / "seat_scores_roberta.csv"
roberta_df.to_csv(OUT_ROBERTA, index=False)

print("=== RoBERTa SEAT Summary ===")
print(f"Total cells scored : {len(roberta_df)}")
print(f"\nBy period:")
for p in ['hist_1920s', 'modern_2020', 'modern_2024']:
    sub = roberta_df[roberta_df['period'] == p]
    if len(sub) > 0:
        male = (sub['seat_career_family'] > 0).sum()
        female = (sub['seat_career_family'] < 0).sum()
        print(f"  {p:<13}: {len(sub):>3} cells | {male} male->career, {female} female->career | "
              f"range [{sub['seat_career_family'].min():.4f}, {sub['seat_career_family'].max():.4f}]")

print(f"\nGlobal baseline (template): {global_seat:.4f}")
print(f"Saved to: {OUT_ROBERTA}")
print(f"\nNote: RoBERTa uses mean-pooling; BERT used [CLS] token.")
print(f"For clean 2x2 comparison, BERT may need re-running with mean-pooling.")

=== RoBERTa SEAT Summary ===
Total cells scored : 108

By period:
  hist_1920s   :   9 cells | 5 male->career, 4 female->career | range [-0.0074, 0.0116]
  modern_2020  :  50 cells | 29 male->career, 20 female->career | range [-0.0092, 0.0090]
  modern_2024  :  49 cells | 30 male->career, 17 female->career | range [-0.0168, 0.0096]

Global baseline (template): -0.0000
Saved to: D:\Shikha\bias_project\outputs\embeddings\roberta\seat_scores_roberta.csv

Note: RoBERTa uses mean-pooling; BERT used [CLS] token.
For clean 2x2 comparison, BERT may need re-running with mean-pooling.
